In [1]:
import os
import sys
from pathlib import Path
import logging
import pandas as pd
from dotenv import load_dotenv
import networkx as nx
from typing import Tuple, Any, List, Dict
from IPython.display import IFrame, display, HTML
import json
import math
# Reduce logging noise
logging.getLogger().setLevel(logging.WARNING)

ROOT_DIR = Path(os.getcwd())
if ROOT_DIR.name == "notebooks":
    ROOT_DIR = ROOT_DIR.parent
sys.path.insert(0, str(ROOT_DIR / "src"))

from graph_generation.sbfl.extractor import ExtractionResult
from visualization.graph_plots import plot_json_graph
from cut_algorithms.boykov_jolly import BoykovJollyCut
from cut_algorithms.data_parsers import read_call_graph
from cut_algorithms.data_parsers import compute_method_tarantula
import evaluation.sbfl_baselines as baselines 
def load_pregen_data(project: str, bug_id: str) -> Tuple[ExtractionResult, Path]:
    """Loads pre-generated extraction data instead of running D4J in real-time."""
    data_dir = ROOT_DIR / "data" / "defects4j" / f"{project}_{bug_id}"
    
    if not data_dir.exists():
        raise FileNotFoundError(f"Pre-generated data not found at {data_dir}")
        
    metrics_csv = data_dir / "sbfl_metrics.csv"
    graph_json = data_dir / "call_graph.json"
    buggy_txt = data_dir / "buggy_methods.txt"
    
    sbfl_metrics = pd.read_csv(metrics_csv).to_dict(orient="records")
    
    with open(buggy_txt, 'r') as f:
        buggy_methods = [line.strip() for line in f if line.strip()]
        
    res = ExtractionResult(
        graph_json=str(graph_json),
        sbfl_metrics=sbfl_metrics,
        buggy_methods=buggy_methods
    )
    
    return res

def json_to_digraph(graph_json: str) -> nx.DiGraph:
    with open(graph_json, 'r') as f:
        graph_data = json.load(f)

    G = nx.DiGraph()
    for edge in graph_data['edges']:
        caller = edge['caller'].replace('$', '.')
        callee = edge['callee'].replace('$', '.')
        G.add_edge(caller, callee, weight=float(edge['frequency']))
    
    for node in graph_data.get('nodes', []):
        n_clean = node.replace('$', '.')
        if n_clean not in G:
            G.add_node(n_clean)
    return G

base_path = ROOT_DIR / "data" / "defects4j"
import ast
df_gt= pd.read_csv("/Users/ashish/master-thesis/Master-Thesis/failure-propagation-and-rca-via-energy-minimization/ground_truth.csv")
def get_buggy_nodes(proj, bugid):
    buggy_nodes = df_gt[df_gt['project']==proj][df_gt['bug_id'] == bugid].buggy_nodes.values[0]
    return ast.literal_eval(buggy_nodes)


def compute_method_tarantula(coverage_df: pd.DataFrame) -> dict[str, float]:
    df = coverage_df
    
    # Aggregate line level to method level
    method_data = {}
    for col in df.columns:
        if col == 'Result':
            continue
        method = map_line_to_method(col)
        if method not in method_data:
            method_data[method] = df[col].copy()
        else:
            method_data[method] = method_data[method] | df[col]
            
    method_df = pd.DataFrame(method_data)
    method_df['Result'] = df['Result']
    
    total_fail = len(method_df[method_df['Result'] == 'Fail'])
    total_pass = len(method_df[method_df['Result'] == 'Pass'])
    
    tarantula_scores = {}
    for method in method_df.columns:
        if method == 'Result':
            continue
        cf = len(method_df[(method_df[method] == 1) & (method_df['Result'] == 'Fail')])
        cp = len(method_df[(method_df[method] == 1) & (method_df['Result'] == 'Pass')])
        
        if total_fail == 0 or (cf == 0 and cp == 0):
            tarantula = 0.0
        else:
            fail_ratio = cf / total_fail
            pass_ratio = cp / total_pass if total_pass > 0 else 0.0
            if fail_ratio + pass_ratio == 0:
                tarantula = 0.0
            else:
                tarantula = fail_ratio / (fail_ratio + pass_ratio)
                
        tarantula_scores[method] = float(tarantula)
        
    return tarantula_scores

def map_line_to_method(line_name: str) -> str:
    """Maps a coverage column like 'com.example$App#process(int):30' to 'com.example.App#process(int)'."""
    if "#" not in line_name:
        return line_name.replace('$', '.')
    return line_name.split(':')[0].replace('$', '.')

def compute_method_ochiai(coverage_df: pd.DataFrame) -> dict[str, float]:
    """
    Computes Ochiai suspiciousness score for each method based on the coverage matrix.
    Aggregates lines into methods using OR logic before computing the score.
    """
    df = coverage_df
    
    # 1. Aggregate line level to method level (Same as your Tarantula code)
    method_data = {}
    for col in df.columns:
        if col == 'Result':
            continue
        method = map_line_to_method(col)
        if method not in method_data:
            method_data[method] = df[col].copy()
        else:
            method_data[method] = method_data[method] | df[col]
            
    method_df = pd.DataFrame(method_data)
    method_df['Result'] = df['Result']
    
    # 2. Global metric: Ochiai only needs total_fail, not total_pass
    total_fail = len(method_df[method_df['Result'] == 'Fail'])
    
    ochiai_scores = {}
    for method in method_df.columns:
        if method == 'Result':
            continue
            
        # 3. Local metrics
        cf = len(method_df[(method_df[method] == 1) & (method_df['Result'] == 'Fail')])
        cp = len(method_df[(method_df[method] == 1) & (method_df['Result'] == 'Pass')])
        
        # 4. Ochiai Math & Edge Case Handling
        # Prevent division by zero if there are no fails or the method is never covered
        if total_fail == 0 or (cf + cp) == 0:
            ochiai = 0.0
        else:
            # The Ochiai Equation
            ochiai = cf / math.sqrt(total_fail * (cf + cp))
                
        ochiai_scores[method] = float(ochiai)
        
    return ochiai_scores


def compute_method_dstar(coverage_df: pd.DataFrame, star: int = 2) -> dict[str, float]:
    """
    Computes D* (D-Star) suspiciousness score for each method based on the coverage matrix.
    Aggregates lines into methods using OR logic before computing the score.
    The 'star' parameter controls the exponent weight (default is 2).
    """
    df = coverage_df
    
    # 1. Aggregate line level to method level
    method_data = {}
    for col in df.columns:
        if col == 'Result':
            continue
        method = map_line_to_method(col)
        if method not in method_data:
            method_data[method] = df[col].copy()
        else:
            method_data[method] = method_data[method] | df[col]
            
    method_df = pd.DataFrame(method_data)
    method_df['Result'] = df['Result']
    
    # 2. Global metric: D* needs total_fail
    total_fail = len(method_df[method_df['Result'] == 'Fail'])
    
    dstar_scores = {}
    for method in method_df.columns:
        if method == 'Result':
            continue
            
        # 3. Local metrics
        cf = len(method_df[(method_df[method] == 1) & (method_df['Result'] == 'Fail')])
        cp = len(method_df[(method_df[method] == 1) & (method_df['Result'] == 'Pass')])
        
        # nf is the number of failing tests that DID NOT execute this method
        nf = total_fail - cf
        denominator = cp + nf
        
        # 4. D* Math & Edge Case Handling
        if denominator == 0:
            if cf > 0:
                dstar = float('inf')
            else:
                dstar = 0.0
        else:
            dstar = (cf ** star) / denominator
                
        dstar_scores[method] = float(dstar)
        
    return dstar_scores

import pandas as pd

def get_rank_list(df, column_name):
    """
    Returns a list of rankings for a given dataframe column.
    Higher scores get lower numerical ranks (e.g., rank 1 for highest score).
    Ties are resolved by assigning the average rank.
    """
    ranks = df[column_name].rank(ascending=False, method='average')
    return ranks.tolist()

def get_baseline_ranked(df, baseline_names):
    for column_name in baseline_names:
        ranks = df[column_name].rank(ascending=False, method='average')
        df[f'{column_name}_rank'] = ranks
    return df

def evaluate(instance, lambd):
    extraction_res = load_pregen_data(project, bug_id)
    G = json_to_digraph(extraction_res.graph_json)

    if len(G.nodes()) == 0:
        raise Exception(f"Empty call graph")

    coverage_csv = instance / "coverage.csv"
    coverage_df = pd.read_csv(coverage_csv)

    method_df = baselines.aggregate_coverage(coverage_df)
    tarantula = baselines.compute_tarantula(method_df)
    ochiai = baselines.compute_ochiai(method_df)
    dstar = baselines.compute_dstar(method_df)

    base = {
        "tarantula": tarantula,
        "ochiai": ochiai,
        "dstar": dstar,
    }

    cut_algo = BoykovJollyCut(G, tarantula, lambd=lambd)
    cut_scores = []
    for node in cut_algo.nodes:
        e0, e1 = cut_algo.compute_min_marginals(node)
        confidence = e0 - e1 # Higher is more buggy
        cut_scores.append({"Method": node, "GraphCut_Score": confidence})
    cut_df = pd.DataFrame(cut_scores)
    buggy_node = get_buggy_nodes(project, int(bug_id))[0]
    print(f"Found buggy node: {buggy_node}")

    
    all_nodes = list(G.nodes())
    nodes_coverage = set(base["tarantula"].keys())
    missing_in_cov= set(all_nodes) - nodes_coverage

    metric_on_nodes: Dict[str, Dict[str, float]] = {
        name: {node: scores.get(node, 0.0) for node in all_nodes}
        for name, scores in base.items()
    }
    df_metrics = pd.DataFrame(metric_on_nodes)
    df_metrics = df_metrics.reset_index().rename(columns={"index": "Method"})
    merged_df = pd.merge(cut_df, df_metrics, on='Method', how='inner')
    merged_df = get_baseline_ranked(merged_df, ['ochiai', 'tarantula', 'dstar', 'GraphCut_Score'])
    return merged_df

import networkx as nx
import pandas as pd

def get_node_properties_df(instance):
    extraction_res = load_pregen_data(project, bug_id)
    G = json_to_digraph(extraction_res.graph_json)
    
    # 1. Compute properties (NetworkX returns dictionaries for these)
    in_degree = dict(G.in_degree())
    out_degree = dict(G.out_degree())
    
    # Centrality metrics
    in_centrality = nx.in_degree_centrality(G)
    out_centrality = nx.out_degree_centrality(G)
    betweenness = nx.betweenness_centrality(G)
    closeness = nx.closeness_centrality(G)
    pagerank = nx.pagerank(G)
    
    # Clustering (local interconnectedness)
    clustering = nx.clustering(G)

    # pandas automatically aligns dictionary keys (nodes) to rows
    df = pd.DataFrame({
        'In_Degree': in_degree,
        'Out_Degree': out_degree,
        'In_Degree_Centrality': in_centrality,
        'Out_Degree_Centrality': out_centrality,
        'Betweenness_Centrality': betweenness,
        'Closeness_Centrality': closeness,
        'PageRank': pagerank,
        'Clustering_Coefficient': clustering
    })
    
    df.index.name = 'Method'
    df = df.reset_index()
    return df

import random
def get_random_sample(num_items):
    return random.sample(bugs, num_items)

def get_graph_properties_df(instance):
    """
    Computes global topological properties for a NetworkX DiGraph
    and returns them as a single-row pandas DataFrame.
    """
    extraction_res = load_pregen_data(project, bug_id)
    G = json_to_digraph(extraction_res.graph_json)
    
    # 1. Basic Size & Structure
    num_nodes = G.number_of_nodes()
    num_edges = G.number_of_edges()
    density = nx.density(G)
    is_dag = nx.is_directed_acyclic_graph(G)
    reciprocity = nx.overall_reciprocity(G)
    
    # 2. Components (Fragmentation)
    num_scc = nx.number_strongly_connected_components(G)
    num_wcc = nx.number_weakly_connected_components(G)
    
    # 3. Assortativity (Wrapped in try/except because it fails on graphs with no edges)
    try:
        degree_assortativity = nx.degree_assortativity_coefficient(G)
    except Exception:
        degree_assortativity = None
        
    # 4. Path Metrics (Safely computed on the largest Strongly Connected Component)
    if num_nodes > 0 and num_scc > 0:
        # Extract the largest strongly connected component
        largest_scc_nodes = max(nx.strongly_connected_components(G), key=len)
        largest_scc = G.subgraph(largest_scc_nodes)
        
        if len(largest_scc) > 1:
            avg_path_length = nx.average_shortest_path_length(largest_scc)
            diameter = nx.diameter(largest_scc)
        else:
            avg_path_length = 0
            diameter = 0
    else:
        avg_path_length = None
        diameter = None

    # 5. Assemble into a dictionary
    data = {
        'Num_Nodes': num_nodes,
        'Num_Edges': num_edges,
        'Density': density,
        'Is_DAG': is_dag,
        'Reciprocity': reciprocity,
        'Num_SCC': num_scc,
        'Num_WCC': num_wcc,
        'Degree_Assortativity': degree_assortativity,
        'Largest_SCC_Avg_Path_Length': avg_path_length,
        'Largest_SCC_Diameter': diameter
    }
    
    return data

In [2]:
bugs = sorted([item for item in base_path.iterdir() if item.is_dir()])

# Bug Selection

In [3]:
len(bugs)

595

In [4]:
# instance_eg = bugs[0]
# project_eg, bug_id_eg = str(instance).split("/")[-1].split("_")
# project_eg, bug_id_eg

In [5]:
sample = get_random_sample(50)
sample

[PosixPath('/Users/ashish/master-thesis/Master-Thesis/failure-propagation-and-rca-via-energy-minimization/data/defects4j/Chart_24'),
 PosixPath('/Users/ashish/master-thesis/Master-Thesis/failure-propagation-and-rca-via-energy-minimization/data/defects4j/Lang_60'),
 PosixPath('/Users/ashish/master-thesis/Master-Thesis/failure-propagation-and-rca-via-energy-minimization/data/defects4j/Mockito_8'),
 PosixPath('/Users/ashish/master-thesis/Master-Thesis/failure-propagation-and-rca-via-energy-minimization/data/defects4j/Jsoup_82'),
 PosixPath('/Users/ashish/master-thesis/Master-Thesis/failure-propagation-and-rca-via-energy-minimization/data/defects4j/Cli_15'),
 PosixPath('/Users/ashish/master-thesis/Master-Thesis/failure-propagation-and-rca-via-energy-minimization/data/defects4j/Gson_5'),
 PosixPath('/Users/ashish/master-thesis/Master-Thesis/failure-propagation-and-rca-via-energy-minimization/data/defects4j/JacksonDatabind_17'),
 PosixPath('/Users/ashish/master-thesis/Master-Thesis/failure-p

In [6]:
# lambdas = [0.1]
# res = []

# for instance in bugs:
#     project, bug_id = str(instance).split("/")[-1].split("_")
#     try:
#         buggy_node = get_buggy_nodes(project, int(bug_id))[0]
#         for lambda_val in lambdas:
#             df_tmp = evaluate(instance, lambda_val)
#             res_tmp = df_tmp[df_tmp['Method'] == buggy_node]
#             print(res_tmp)
#             data_tmp = {
#                 "project": project,
#                 "bug_id": bug_id,
#                 "lambda_val":lambda_val,
#                 "results": df_tmp,
#                 "topological_properties": get_node_properties_df(instance),
#                 "network_properties": get_graph_properties_df(instance),
#                 "buggy_node": buggy_node
#             }
#             res.append(data_tmp)
#     except Exception as e:
#         print(e)

In [7]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

def compile_analysis_df(res_list):
    """Flattens the nested results list into a single DataFrame for the buggy node."""
    records = []
    for item in res_list:
        b_node = item['buggy_node']
        
        # 1. Extract Buggy Node Results (Scores & Ranks)
        res_df = item['results']
        b_results = res_df[res_df['Method'] == b_node]
        
        if b_results.empty: 
            continue
        b_results = b_results.iloc[0]
        
        # 2. Extract Buggy Node Topological Properties
        topo_df = item['topological_properties']
        b_topo = topo_df[topo_df['Method'] == b_node] 
        if b_topo.empty: 
            continue
        b_topo = b_topo.iloc[0]
        
        # 3. Extract Graph Level Properties
        net_dict = item['network_properties'] 
        
        # 4. Combine into one flat record
        record = {
            'project': item['project'],
            'bug_id': item['bug_id'],
            'lambda_val': item['lambda_val'],
            'tarantula': b_results['tarantula'],
            'ochiai': b_results['ochiai'],
            'dstar': b_results['dstar'],
            'GraphCut_Score': b_results['GraphCut_Score'],
            'GraphCut_Rank': b_results.get('GraphCut_Score_rank', None),
            'tarantula_rank': b_results['tarantula_rank'],
            'ochiai_rank': b_results['ochiai_rank'],
            'dstar_rank': b_results['dstar_rank'],
            'rank_gain_tarantula':b_results['tarantula_rank'] - b_results['GraphCut_Score_rank'],
            # 'exam_gain_tarantula':b_results['tarantula_exam'] - b_results['GraphCut_Score'] 
        }
        
        # Add all topological metrics dynamically
        for col in topo_df.columns:
            if col != 'Method':  # Skip the key column
                record[col] = b_topo[col]
            
        # Add all network metrics dynamically from the dictionary
        for key, val in net_dict.items():
            record[key] = val
            
        records.append(record)
        
    return pd.DataFrame(records)

# Create the master dataframe for plotting
# analysis_df = compile_analysis_df(res)
# analysis_df

In [8]:
# analysis_df[analysis_df['lambda_val']==0.1].rank_gain_tarantula

In [9]:
# analysis_df[analysis_df['lambda_val']==1.0].rank_gain_tarantula.mean()

In [10]:
def plot_baselines_vs_node_topo(df, topo_metric='In_Degree'):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    baselines = ['tarantula', 'ochiai', 'dstar']
    
    for ax, baseline in zip(axes, baselines):
        sns.scatterplot(data=df, x=topo_metric, y=baseline, ax=ax, alpha=0.7)
        sns.regplot(data=df, x=topo_metric, y=baseline, ax=ax, scatter=False, color='red')
        
        ax.set_title(f'{baseline.capitalize()} vs {topo_metric}')
        ax.set_xlabel(topo_metric.replace('_', ' '))
        ax.set_ylabel(baseline.capitalize() + ' Score')
        
    plt.tight_layout()
    plt.show()


# plot_baselines_vs_node_topo(analysis_df, topo_metric='PageRank')

In [11]:
def plot_gc_score_vs_graph_topo(df, graph_metric='Density'):
    plt.figure(figsize=(10, 6))
    
    sns.scatterplot(data=df, x=graph_metric, y='GraphCut_Score', 
                    hue='lambda_val', palette='viridis', s=100, alpha=0.8)
    
    plt.title(f'GraphCut Score vs {graph_metric.replace("_", " ")}', fontsize=14)
    plt.xlabel(graph_metric.replace('_', ' '), fontsize=12)
    plt.ylabel('GraphCut Score', fontsize=12)
    plt.legend(title='Lambda (λ)')
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.show()

# Example usage:
# plot_gc_score_vs_graph_topo(analysis_df, graph_metric='Degree_Assortativity')

In [12]:
def plot_gc_rank_vs_node_topo(df, topo_metric='In_Degree'):
    plt.figure(figsize=(10, 6))
    
    sns.scatterplot(data=df, x=topo_metric, y='GraphCut_Rank', 
                    hue='lambda_val', palette='plasma', s=100, alpha=0.8)
    
    # Invert Y-axis so rank 1 is at the top
    plt.gca()
    
    plt.title(f'GraphCut Rank vs {topo_metric.replace("_", " ")}', fontsize=14)
    plt.xlabel(topo_metric.replace('_', ' '), fontsize=12)
    plt.ylabel('Rank (1 is best)', fontsize=12)
    plt.legend(title='Lambda (λ)')
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.show()

# Example usage:
# plot_gc_rank_vs_node_topo(analysis_df, topo_metric='Out_Degree')

In [13]:
# plot_gc_rank_vs_node_topo(analysis_df, topo_metric='Betweenness_Centrality')

In [14]:
# analysis_df[analysis_df['lambda_val']==0.1][['tarantula_rank', 'GraphCut_Rank', 'rank_gain_tarantula',]]

In [15]:
# analysis_df[analysis_df['lambda_val']==0.1][['project','bug_id', 'tarantula_rank', 'GraphCut_Rank', 'rank_gain_tarantula',]].sort_values(by='rank_gain_tarantula')

In [16]:
# lambdas = [0.1]
# res_10 = {}
# res = {}
# for instance in sample:
#     project, bug_id = str(instance).split("/")[-1].split("_")
#     print(project, bug_id)
#     try:
#         buggy_node = get_buggy_nodes(project, int(bug_id))[0]
#         for lambda_val in lambdas:
#             df_tmp = evaluate(instance, lambda_val)
#             res_tmp = df_tmp[df_tmp['Method'] == buggy_node]
#             print(res_tmp)
#             res[f"{project}_{bug_id}_{lambda_val}"] = res_tmp
#     except Exception as e:
#         print(e)

In [17]:
# project = 'Jsoup'
# bug_id = '63'
# instance = Path('/Users/ashish/master-thesis/Master-Thesis/failure-propagation-and-rca-via-energy-minimization/data/defects4j/Jsoup_63')
# print(instance)
# buggy_node = get_buggy_nodes(project, int(bug_id))[0]
# print(buggy_node)
# df_tmp = evaluate(instance, 0.1)
# res_tmp = df_tmp[df_tmp['Method'] == buggy_node]
# print(res_tmp)

In [18]:
# project = 'Jsoup'
# bug_id = '63'
# print(buggy_node)
# df_tmp = evaluate(instance, 0.01)
# res_tmp = df_tmp[df_tmp['Method'] == buggy_node]
# print(res_tmp)

# Visualizations

In [19]:
import pandas as pd
import matplotlib.pyplot as plt

def plot_rank_gain_correlation(df, feature_cols, target_col='rank_gain_tarantula'):
    """
    Calculates and plots the Spearman correlation of a list of features 
    against a target metric (like rank gain).
    """
    # 1. Ensure the target column exists. 
    # Positive gain = GraphCut rank is a smaller number (better) than Tarantula rank
    if target_col not in df.columns:
        print(f"'{target_col}' not found. Calculating as 'tarantula_rank' - 'GraphCut_Rank'...")
        df[target_col] = df['tarantula_rank'] - df['GraphCut_Rank']

    # 2. Calculate Spearman correlations for the provided list of columns
    # We dropna() just for the calculation to ensure missing values don't break the math
    correlations = df[feature_cols + [target_col]].corr(method='spearman')[target_col]
    correlations = correlations.drop(target_col).sort_values()

    # 3. Create the plot
    # Dynamically scale height based on how many features you pass in
    plt.figure(figsize=(10, max(5, len(feature_cols) * 0.5)))
    
    # Green for positive correlation, Red for negative
    colors = ['#d62728' if x < 0 else '#2ca02c' for x in correlations.values]
    
    bars = plt.barh(correlations.index, correlations.values, color=colors, alpha=0.8)
    
    # 4. Formatting
    plt.axvline(0, color='black', linewidth=1.2)
    plt.title(f"Impact of Features on {target_col}\n(Spearman Correlation)", fontsize=14, pad=15)
    plt.xlabel("Correlation Coefficient (ρ)", fontsize=12)
    plt.grid(axis='x', linestyle='--', alpha=0.6)
    
    # Add exact numeric values next to the bars for easy reading
    for bar, val in zip(bars, correlations.values):
        # Push the text slightly left or right of the bar edge depending on direction
        offset = 0.02 if val >= 0 else -0.02
        ha = 'left' if val >= 0 else 'right'
        plt.text(val + offset, bar.get_y() + bar.get_height() / 2, 
                 f"{val:.2f}", va='center', ha=ha, fontsize=10, fontweight='bold')
                 
    # Extend x-axis limits slightly so the text labels don't get cut off
    x_min, x_max = plt.xlim()
    plt.xlim(x_min - 0.05, x_max + 0.05)
                 
    plt.tight_layout()
    plt.show()

# ==========================================
# Example Usage:
# ==========================================
# List the exact column names you want to test from your dataframe
features_to_test = [
    'Num_Nodes', 'Num_Edges', 'Density', 'Reciprocity', 
    'In_Degree', 'Out_Degree', 'PageRank', 'Betweenness_Centrality'
]

# plot_rank_gain_correlation(analysis_df, features_to_test)

In [20]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

def plot_correlation_by_lambda(df, feature_cols, target_col='rank_gain_tarantula'):
    """
    Calculates and plots the Spearman correlation of a list of features 
    against a target metric, grouped by the lambda value.
    """
    # 1. Ensure the target column exists
    if target_col not in df.columns:
        print(f"'{target_col}' not found. Calculating as 'tarantula_rank' - 'GraphCut_Rank'...")
        df[target_col] = df['tarantula_rank'] - df['GraphCut_Rank']
        
    # 2. Calculate correlations separately for each lambda
    corr_data = []
    lambdas = sorted(df['lambda_val'].unique())
    
    for l_val in lambdas:
        # Filter dataframe for this specific lambda run
        df_lambda = df[df['lambda_val'] == l_val]
        
        # Calculate Spearman correlations safely
        corrs = df_lambda[feature_cols + [target_col]].corr(method='spearman')[target_col]
        
        # Store results in a flat list of dictionaries for Seaborn
        for feature in feature_cols:
            corr_data.append({
                'Feature': feature,
                'Lambda': f"λ = {l_val}",
                'Correlation': corrs[feature]
            })
            
    # Convert to a DataFrame for easy plotting
    corr_df = pd.DataFrame(corr_data)
    
    # Sort features by their *average* correlation across all lambdas to keep the plot neat
    avg_corrs = corr_df.groupby('Feature')['Correlation'].mean().sort_values()
    
    # 3. Create the grouped bar plot
    # Dynamically scale height based on number of features
    plt.figure(figsize=(10, max(6, len(feature_cols) * 0.8)))
    
    # Seaborn handles the side-by-side grouping automatically via the 'hue' parameter
    ax = sns.barplot(
        data=corr_df, 
        y='Feature', 
        x='Correlation', 
        hue='Lambda', 
        order=avg_corrs.index,
        palette='Set2' # Distinct colors for different lambdas
    )
    
    # 4. Formatting and aesthetics
    plt.axvline(0, color='black', linewidth=1.2)
    plt.title(f"Feature Impact on {target_col} by Lambda\n(Spearman Correlation)", fontsize=14, pad=15)
    plt.xlabel("Correlation Coefficient (ρ)", fontsize=12)
    plt.ylabel("") # Clear the y-axis label since feature names are self-explanatory
    plt.grid(axis='x', linestyle='--', alpha=0.6)
    
    # Add a clean legend
    plt.legend(title="Structural Smoothing", bbox_to_anchor=(1.05, 1), loc='upper left')
    
    plt.tight_layout()
    plt.show()

# ==========================================
# Example Usage:
# ==========================================
features_to_test = [
    'Num_Nodes', 'Num_Edges', 'Density', 'Reciprocity', 
    'In_Degree', 'Out_Degree', 'PageRank', 'Betweenness_Centrality'
]

# plot_correlation_by_lambda(analysis_df, features_to_test)

In [21]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import math

def plot_gain_vs_graph_metrics(df, graph_metrics, target_col='rank_gain_tarantula'):
    """
    Plots the target rank gain against a list of graph-wide metrics in a grid.
    Includes a baseline at Y=0 and trendlines separated by lambda_val.
    """
    # Calculate grid size (3 columns wide)
    cols = 3
    rows = math.ceil(len(graph_metrics) / cols)
    
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4 * rows))
    axes = axes.flatten() # Flatten to easily index in a loop
    
    # Define a clean color palette for our lambdas
    lambdas = sorted(df['lambda_val'].dropna().unique())
    colors = sns.color_palette('Set1', n_colors=len(lambdas))
    
    for i, metric in enumerate(graph_metrics):
        ax = axes[i]
        
        # 1. Scatter plot of the actual data points
        sns.scatterplot(
            data=df, x=metric, y=target_col, 
            hue='lambda_val', palette='Set1', 
            alpha=0.6, s=60, ax=ax, edgecolor=None
        )
        
        # 2. Add trendlines for each lambda to clearly see the divergence
        for j, l_val in enumerate(lambdas):
            subset = df[df['lambda_val'] == l_val]
            # Only draw a trendline if we have enough points and the metric isn't completely null
            if len(subset) > 1 and subset[metric].notnull().sum() > 1:
                sns.regplot(
                    data=subset, x=metric, y=target_col,
                    scatter=False, ax=ax, color=colors[j],
                    line_kws={'linestyle': '--', 'linewidth': 2}
                )
                
        # 3. Add the Breakeven Line (Y=0)
        # Above this line: GraphCut Wins. Below this line: Tarantula Wins.
        ax.axhline(0, color='black', linestyle='-', linewidth=1.5, zorder=0)
        
        # 4. Formatting
        ax.set_title(f'Rank Gain vs {metric.replace("_", " ")}', fontsize=12, fontweight='bold')
        ax.set_xlabel(metric.replace('_', ' '), fontsize=10)
        ax.set_ylabel('Rank Gain (Higher = Better)', fontsize=10)
        
        # Keep legend only on the first plot to avoid clutter
        if i == 0:
            ax.legend(title='Lambda (λ)')
        else:
            if ax.get_legend() is not None:
                ax.get_legend().remove()

    # Hide any unused subplots if the list length isn't a perfect multiple of 3
    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])
        
    plt.tight_layout()
    plt.show()

# ==========================================
# Example Usage:
# ==========================================
# List the graph-level metrics you extracted earlier
graph_metrics_to_test = [
    'Num_Nodes', 'Num_Edges', 'Density', 
    'Reciprocity', 'Degree_Assortativity', 'Largest_SCC_Avg_Path_Length'
]
plt.figure(figsize=(28,28))
# plot_gain_vs_graph_metrics(analysis_df, graph_metrics_to_test)

<Figure size 2800x2800 with 0 Axes>

<Figure size 2800x2800 with 0 Axes>

project	bug_id	tarantula_rank	GraphCut_Rank	rank_gain_tarantula
91	Compress	43	77.0	168.0	-91.0
89	Compress	41	70.0	94.0	-24.0
119	Gson	2	28.0	49.0	-21.0
85	Compress	35	21.0	30.5	-9.5
63	Compress	12	6.5	16.0	-9.5
156	JacksonDatabind	19	30.0	37.0	-7.0
111 Chart 26 	98.5  78.0	20.5
113 Codec 14 	18.5	1.0		27.5
	


In [22]:
lambdas = [0.1]
res = []
base_path = Path('/Users/ashish/master-thesis/Master-Thesis/failure-propagation-and-rca-via-energy-minimization/data/defects4j')

bugs = [
    ('Compress', 43),
    ('Compress', 41),
    ('Gson', 2),
    ('Compress', 35),
    ('Compress', 12),
    ('JacksonDatabind', 19),
    ('Chart', 26),
    ('Codec', 14),
       ]
for project,bug_id in bugs:
    instance = base_path / f"{project}_{bug_id}"
    print(project, bug_id, instance)
    project, bug_id = str(instance).split("/")[-1].split("_")
    try:
        buggy_node = get_buggy_nodes(project, int(bug_id))[0]
        for lambda_val in lambdas:
            df_tmp = evaluate(instance, lambda_val)
            res_tmp = df_tmp[df_tmp['Method'] == buggy_node]
            # print(res_tmp)
            data_tmp = {
                "project": project,
                "bug_id": bug_id,
                "lambda_val":lambda_val,
                "results": df_tmp,
                "topological_properties": get_node_properties_df(instance),
                "network_properties": get_graph_properties_df(instance),
                "buggy_node": buggy_node
            }
            res.append(data_tmp)
    except Exception as e:
        print(e)

Compress 43 /Users/ashish/master-thesis/Master-Thesis/failure-propagation-and-rca-via-energy-minimization/data/defects4j/Compress_43


/var/folders/zp/3681gr8x5mg9vt0z3xbtlzx00000gn/T/ipykernel_6447/3927014183.py:70: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  buggy_nodes = df_gt[df_gt['project']==proj][df_gt['bug_id'] == bugid].buggy_nodes.values[0]
/var/folders/zp/3681gr8x5mg9vt0z3xbtlzx00000gn/T/ipykernel_6447/3927014183.py:70: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  buggy_nodes = df_gt[df_gt['project']==proj][df_gt['bug_id'] == bugid].buggy_nodes.values[0]
/var/folders/zp/3681gr8x5mg9vt0z3xbtlzx00000gn/T/ipykernel_6447/3927014183.py:70: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  buggy_nodes = df_gt[df_gt['project']==proj][df_gt['bug_id'] == bugid].buggy_nodes.values[0]


Found buggy node: org.apache.commons.compress.archivers.zip.ZipArchiveOutputStream#createLocalFileHeader(org.apache.commons.compress.archivers.zip.ZipArchiveEntry,java.nio.ByteBuffer,boolean,boolean,long)
Compress 41 /Users/ashish/master-thesis/Master-Thesis/failure-propagation-and-rca-via-energy-minimization/data/defects4j/Compress_41


/var/folders/zp/3681gr8x5mg9vt0z3xbtlzx00000gn/T/ipykernel_6447/3927014183.py:70: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  buggy_nodes = df_gt[df_gt['project']==proj][df_gt['bug_id'] == bugid].buggy_nodes.values[0]
/var/folders/zp/3681gr8x5mg9vt0z3xbtlzx00000gn/T/ipykernel_6447/3927014183.py:70: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  buggy_nodes = df_gt[df_gt['project']==proj][df_gt['bug_id'] == bugid].buggy_nodes.values[0]


Found buggy node: org.apache.commons.compress.archivers.zip.ZipArchiveInputStream#getNextZipEntry()
Gson 2 /Users/ashish/master-thesis/Master-Thesis/failure-propagation-and-rca-via-energy-minimization/data/defects4j/Gson_2


...

/var/folders/zp/3681gr8x5mg9vt0z3xbtlzx00000gn/T/ipykernel_6447/3927014183.py:70: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  buggy_nodes = df_gt[df_gt['project']==proj][df_gt['bug_id'] == bugid].buggy_nodes.values[0]


Found buggy node: com.fasterxml.jackson.databind.type.TypeFactory#_mapType(java.lang.Class)
Chart 26 /Users/ashish/master-thesis/Master-Thesis/failure-propagation-and-rca-via-energy-minimization/data/defects4j/Chart_26


/var/folders/zp/3681gr8x5mg9vt0z3xbtlzx00000gn/T/ipykernel_6447/3927014183.py:70: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  buggy_nodes = df_gt[df_gt['project']==proj][df_gt['bug_id'] == bugid].buggy_nodes.values[0]
/var/folders/zp/3681gr8x5mg9vt0z3xbtlzx00000gn/T/ipykernel_6447/3927014183.py:70: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  buggy_nodes = df_gt[df_gt['project']==proj][df_gt['bug_id'] == bugid].buggy_nodes.values[0]
/var/folders/zp/3681gr8x5mg9vt0z3xbtlzx00000gn/T/ipykernel_6447/3927014183.py:70: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  buggy_nodes = df_gt[df_gt['project']==proj][df_gt['bug_id'] == bugid].buggy_nodes.values[0]


Found buggy node: org.jfree.chart.axis.Axis#drawLabel(java.lang.String,java.awt.Graphics2D,java.awt.geom.Rectangle2D,java.awt.geom.Rectangle2D,org.jfree.chart.util.RectangleEdge,org.jfree.chart.axis.AxisState,org.jfree.chart.plot.PlotRenderingInfo)
Codec 14 /Users/ashish/master-thesis/Master-Thesis/failure-propagation-and-rca-via-energy-minimization/data/defects4j/Codec_14
Found buggy node: org.apache.commons.codec.language.bm.PhoneticEngine#applyFinalRules(org.apache.commons.codec.language.bm.PhoneticEngine.PhonemeBuilder,java.util.Map)


/var/folders/zp/3681gr8x5mg9vt0z3xbtlzx00000gn/T/ipykernel_6447/3927014183.py:70: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  buggy_nodes = df_gt[df_gt['project']==proj][df_gt['bug_id'] == bugid].buggy_nodes.values[0]


In [23]:
analysis_df = compile_analysis_df(res)
analysis_df

,project,bug_id,lambda_val,tarantula,ochiai,dstar,GraphCut_Score,GraphCut_Rank,tarantula_rank,ochiai_rank,...,Num_Nodes,Num_Edges,Density,Is_DAG,Reciprocity,Num_SCC,Num_WCC,Degree_Assortativity,Largest_SCC_Avg_Path_Length,Largest_SCC_Diameter
0,Compress,43,0.1,0.240210,0.019518,0.007812,-1.779379,168.0,77.0,77.0,...,235,375,0.006819,True,0.000000,235,13,-0.097317,0.000000,0
1,Compress,41,0.1,0.603189,0.067344,0.063492,-0.214776,94.0,70.0,65.0,...,271,416,0.005685,False,0.004808,270,8,-0.125887,1.000000,1
2,Gson,2,0.1,0.607547,0.040000,0.001603,0.281647,49.0,28.0,28.0,...,375,674,0.004806,False,0.008902,363,9,-0.182673,3.750000,8
3,Compress,35,0.1,0.847856,0.094491,0.017857,1.743657,30.5,21.0,21.0,...,138,164,0.008674,True,0.000000,138,9,-0.311510,0.000000,0
4,Compress,12,0.1,0.943114,0.223607,0.052632,2.714219,16.0,6.5,6.5,...,67,72,0.016282,True,0.000000,67,3,-0.224846,0.000000,0
5,JacksonDatabind,19,0.1,0.695829,0.130537,0.451493,0.924985,37.0,30.0,406.0,...,848,1387,0.001931,False,0.010094,824,20,-0.108961,3.060606,7
6,Chart,26,0.1,0.985388,0.699206,21.043478,4.330880,78.0,98.5,62.5,...,543,797,0.002708,False,0.002509,537,29,-0.205023,2.500000,5
7,Codec,14,0.1,0.967638,0.408248,0.800000,4.058461,1.0,28.5,28.5,...,87,94,0.012563,False,0.021277,86,10,-0.114123,1.000000,1
